# TESS FFI exoplanet search walkthrough

Runnable on **Google Colab** or **Kaggle** (CPU). Install deps, then walk through the same steps as `SKILL.md`.

**Worked example:** TIC 370612437 (community FFI candidate, P ≈ 3.56 d).  
**Sanity check:** TOI-700 b (TIC 150428135, P ≈ 9.98 d) with BLS restricted to 8–12 d.

In [ ]:
!pip install -q lightkurve astroquery wotan matplotlib pandas scipy
import sys
from pathlib import Path
ROOT = Path('..').resolve() if (Path('..') / 'exoplanet_hunter').exists() else Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import json
from exoplanet_hunter import lc_utils

## 1. Download light curves (SPOC + QLP)

In [ ]:
CANDIDATE = 370612437
KNOWN = 150428135
cache = ROOT / 'data' / 'mast_cache'
lc_path = ROOT / 'data' / 'lightcurves' / f'{CANDIDATE}.npz'
lc_path.parent.mkdir(parents=True, exist_ok=True)
meta = lc_utils.fetch_tic_lightcurves(CANDIDATE, out_path=lc_path, cache_dir=cache)
print('Sectors:', sorted(set(int(s) for s in meta['sector'])), 'points:', len(meta['time']))

## 2. Detrend and BLS search

In [ ]:
lc = lc_utils.load_npz(lc_path)
lc, bad = lc_utils.reject_bad_sectors(lc)
cleaned = lc_utils.bin_lc(lc_utils.clean_lc(lc))
search = lc_utils.bls_search(cleaned, rho_sun=1.0)
print(json.dumps(search, indent=2))

In [ ]:
import matplotlib.pyplot as plt
ph = lc_utils.phase_fold(cleaned['time'], search['period_d'], search['epoch_btjd'])
plt.figure(figsize=(8,3))
plt.scatter(ph, cleaned['flux'], s=1, alpha=0.4)
plt.xlabel('Phase')
plt.ylabel('Flux')
plt.title(f'TIC {CANDIDATE}  P={search["period_d"]:.5f} d')
plt.show()

## 3. Vet (subprocess to CLI)

In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'search_transits.py'), str(CANDIDATE)], check=True)
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'vet_candidate.py'), str(CANDIDATE)], check=False)

## 4. Sanity check — known TOI-700 b

In [ ]:
known_path = ROOT / 'data' / 'lightcurves' / f'{KNOWN}.npz'
if not known_path.exists():
    lc_utils.fetch_tic_lightcurves(KNOWN, out_path=known_path, cache_dir=cache)
klc = lc_utils.load_npz(known_path)
klc, _ = lc_utils.reject_bad_sectors(klc)
kc = lc_utils.bin_lc(lc_utils.clean_lc(klc))
ksearch = lc_utils.bls_search(kc, period_min=8, period_max=12)
published = 9.97821
diff = abs(ksearch['period_d'] / published - 1)
print(f'Measured P={ksearch["period_d"]:.5f} d  (published {published})  diff={diff*100:.2f}%')
assert diff < 0.01, 'TOI-700 b period should recover within 1% when windowed'

## 5. Predict future transits & CTOI fields

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'predict_transits.py'), str(CANDIDATE)], check=True)
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'make_ctoi_report.py'), str(CANDIDATE)], check=True)
print('See docs/SUBMITTING_TO_NASA.md for ExoFOP upload steps.')